# Benchmark: Reproducing SI WIMP limits from XENONnT, LZ, and PandaX-4T

In [ ]:
import os
import diamx
from diamx.model import DiamxModel
from diamx.utils import generate_bin_array
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

diamx_dir = os.path.dirname(diamx.__file__)

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="appletree.config")
warnings.filterwarnings("ignore", category=UserWarning, module="appletree.component")

Some plot styles...

In [ ]:
plt.style.use("seaborn-v0_8-colorblind")
plt.rcParams.update(
    {
        "figure.figsize": (6.4, 4.8),
        "figure.dpi": 600,
        "font.family": "serif",
        "font.size": 12,
        # 'figure.dpi': 300,
        "lines.linewidth": 2.0,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "xtick.major.size": 8,
        "xtick.minor.size": 4,
        "ytick.major.size": 8,
        "ytick.minor.size": 4,
        "xtick.major.width": 1,
        "xtick.minor.width": 0.7,
        "ytick.major.width": 1,
        "ytick.minor.width": 0.7,
        "xtick.direction": "in",
        "ytick.direction": "in",
        # 'legend.loc': 'upper center',
        # 'legend.bbox_to_anchor': (0.5, 1.05),
        # 'legend.ncol': 3,
        "legend.fancybox": True,  # if True, use a rounded box for the
        # legend background, else a rectangle
        "legend.fontsize": 12,
    }
)

## 0. Generate Input WIMP spectrum

Using package wimprates

In [ ]:
import wimprates

energies = np.linspace(0.01, 100, 200)
wimp_masses = (
    list(range(6, 130, 2)) + list(range(130, 400, 10)) + [400, 500, 600, 800, 1000]
)
wimp_masses += list(range(100, 1000, 50)) + [1000, 2500, 5000, 10000]
wimp_masses = set(wimp_masses)
spectrum_data = np.zeros((len(energies), 2))
spectrum_data[:, 0] = energies
os.makedirs("./diamx_output/spectrum", exist_ok=True)
for wimp_mass in wimp_masses:
    spectrum_csv_fpath = os.path.join(
        "./diamx_output", "spectrum", f"wimp_{wimp_mass}GeV.csv"
    )
    spectrum_data[:, 1] = wimprates.rate_wimp_std(
        energies, mw=wimp_mass, sigma_nucleon=1e-44
    )
    np.savetxt(spectrum_csv_fpath, spectrum_data, delimiter=",")

In [ ]:
def plot_contours_and_pies(context, plot_config, file_base="../diamx/data"):
    """
    Plots the contours for the given plot configuration.

    Parameters
    ----------
    context : DiamxModel
        The Diamx model context.
    plot_config : dict
        The plot configuration dictionary containing the experiment name and specifications for each plot item.
        It should have the following structure:
        {
            "experiment_name": "experiment_name",
            "signal_parameter_value": 1000, # For the best_fit model
            "content_specs": [
                # For each component to plot
                {
                    "type": "bkg", # or "shaped_bkg" or "best_fit_bkg" or "signal"
                    "plot_contour_diamx": "contour", # or "contourf" or False
                    "plot_contour_literature": True, # or False
                    "include_in_pie_charts": True,
                    "component_name": "background_name",
                    "shape_parameter_value": 0.5, # Only for shaped_bkg
                    "color": "blue",
                    "one_sigma_contour_fname": "one_sigma_contour.csv",
                    "two_sigma_contour_fname": "two_sigma_contour.csv",
                    "log10_contour": True,
                    "label": "Label for the plot item"
                },
                ...
            ],
            "fig_specs": {
                "figure_size": [4, 3],
                "axis_names": ["cs1", "cs2"],
                "axis_labels": ["cS1 [PE]", "cS2 [PE]"],
                "axis_limits": [[0, 100], [10**2.1, 10**4.1]],
                "axis_scales": ["linear", "log"],
                "contour_linestyles": ["--", "-"],
                "plot_pie_charts": True,
                "pie_size_min": 2,
                "pie_size_max": 10,
                "pie_axis_width": 0.01,
                "pie_axis_height": 0.01,
                "legend_kwargs": {
                    "loc": "upper center",
                    "bbox_to_anchor": (0.5, 1.05),
                    "ncol": 3,
                }
            },
        }
    file_base : str
        The base directory where the contour files are located.

    Returns
    -------
    fig : matplotlib.figure.Figure
        The matplotlib figure object.
    ax : matplotlib.axes.Axes
        The matplotlib axes object.
    """
    fig_specs = plot_config.get(
        "fig_specs",
        {
            "figure_size": [4, 3],
            "axis_names": ["cs1", "cs2"],
            "axis_labels": ["cS1 [PE]", "cS2 [PE]"],
            "axis_limits": [[0, 100], [10**2.1, 10**4.1]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 2,
            "pie_size_max": 10,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
        },
    )
    fig, ax = plt.subplots(1, 1, figsize=fig_specs.get("figure_size", [4, 3]))
    ax.set_xlabel(fig_specs["axis_labels"][0])
    ax.set_ylabel(fig_specs["axis_labels"][1])
    ax.set_xscale(fig_specs["axis_scales"][0])
    ax.set_xlim(fig_specs["axis_limits"][0])
    ax.set_yscale(fig_specs["axis_scales"][1])
    ax.set_ylim(fig_specs["axis_limits"][1])
    experiment_name = plot_config["experiment_name"]
    for plot_item in plot_config["content_specs"]:
        plot_mode_diamx = plot_item.get("plot_contour_diamx", "contour")
        if plot_mode_diamx is not False:
            if plot_item["type"] == "bkg":
                if plot_mode_diamx == "contour":
                    context.plot_bkg_template(
                        experiment_name,
                        plot_item["component_name"],
                        mode=["contour"],
                        contour_kwargs={
                            "colors": [plot_item["color"], plot_item["color"]],
                            "linestyles": fig_specs.get(
                                "contour_linestyles", ["--", "-"]
                            ),
                        },
                    )
                elif plot_mode_diamx == "contourf":
                    context.plot_bkg_template(
                        experiment_name,
                        plot_item["component_name"],
                        mode=["contourf"],
                        contourf_kwargs={
                            "colors": [plot_item["color"], plot_item["color"]],
                            "alpha": fig_specs.get("contourf_alpha", [0.1, 0.2]),
                            "extend": "max",
                        },
                    )
            elif plot_item["type"] == "shaped_bkg":
                assert (
                    "shape_parameter_value" in plot_item
                ), "shape_parameter_value must be provided for shaped_bkg"
                if plot_mode_diamx == "contour":
                    context.plot_bkg_template(
                        experiment_name,
                        plot_item["component_name"],
                        shape_parameter_value=plot_item["shape_parameter_value"],
                        mode=["contour"],
                        contour_kwargs={
                            "colors": [plot_item["color"], plot_item["color"]],
                            "linestyles": fig_specs.get(
                                "contour_linestyles", ["--", "-"]
                            ),
                        },
                    )
                elif plot_mode_diamx == "contourf":
                    context.plot_bkg_template(
                        experiment_name,
                        plot_item["component_name"],
                        shape_parameter_value=plot_item["shape_parameter_value"],
                        mode=["contourf"],
                        contourf_kwargs={
                            "colors": [plot_item["color"], plot_item["color"]],
                            "alpha": fig_specs.get("contourf_alpha", [0.1, 0.2]),
                            "extend": "max",
                        },
                    )
            elif plot_item["type"] == "best_fit_bkg":
                if plot_mode_diamx == "contour":
                    context.plot_best_fit_bkg_mh(
                        experiment_name,
                        plot_config["signal_parameter_value"],
                        bkg_to_include=plot_item.get("bkg_to_include", None),
                        mode=["contour"],
                        contour_kwargs={
                            "colors": [plot_item["color"], plot_item["color"]],
                            "linestyles": fig_specs.get(
                                "contour_linestyles", ["--", "-"]
                            ),
                        },
                    )
                elif plot_mode_diamx == "contourf":
                    context.plot_best_fit_bkg_mh(
                        experiment_name,
                        plot_config["signal_parameter_value"],
                        bkg_to_include=plot_item.get("bkg_to_include", None),
                        mode=["contourf"],
                        contourf_kwargs={
                            "colors": [plot_item["color"], plot_item["color"]],
                            "alpha": fig_specs.get("contourf_alpha", [0.1, 0.2]),
                            "extend": "max",
                        },
                    )
            elif plot_item["type"] == "signal":
                context.plot_signal_template(
                    experiment_name,
                    plot_config["signal_parameter_value"],
                    mode=["contour"],
                    contour_kwargs={
                        "colors": [plot_item["color"], plot_item["color"]],
                        "linestyles": ["--", "-"],
                    },
                )
            else:
                raise ValueError(f"Unknown plot item type: {plot_item['type']}")
        if plot_item.get("plot_contour_literature", True):
            if "one_sigma_contour_fname" in plot_item:
                if isinstance(plot_item["one_sigma_contour_fname"], str):
                    one_sigma_fnames = [plot_item["one_sigma_contour_fname"]]
                else:
                    one_sigma_fnames = plot_item["one_sigma_contour_fname"]
                for one_sigma_fname in one_sigma_fnames:
                    one_sigma_contour = np.loadtxt(
                        os.path.join(file_base, one_sigma_fname), delimiter=","
                    )
                    if plot_item.get("log10_contour", False):
                        ax.fill(
                            one_sigma_contour[:, 0],
                            10 ** one_sigma_contour[:, 1],
                            color=plot_item["color"],
                            alpha=0.2,
                        )
                    else:
                        ax.fill(
                            one_sigma_contour[:, 0],
                            one_sigma_contour[:, 1],
                            color=plot_item["color"],
                            alpha=0.2,
                        )
            if "two_sigma_contour_fname" in plot_item:
                if isinstance(plot_item["two_sigma_contour_fname"], str):
                    two_sigma_fnames = [plot_item["two_sigma_contour_fname"]]
                else:
                    two_sigma_fnames = plot_item["two_sigma_contour_fname"]
                for two_sigma_fname in two_sigma_fnames:
                    two_sigma_contour = np.loadtxt(
                        os.path.join(file_base, two_sigma_fname), delimiter=","
                    )
                    if plot_item.get("log10_contour", False):
                        ax.fill(
                            two_sigma_contour[:, 0],
                            10 ** two_sigma_contour[:, 1],
                            color=plot_item["color"],
                            alpha=0.1,
                        )
                    else:
                        ax.fill(
                            two_sigma_contour[:, 0],
                            two_sigma_contour[:, 1],
                            color=plot_item["color"],
                            alpha=0.1,
                        )
    fig.canvas.draw()  # Render the figure to get the correct transforms
    data_to_figure = ax.transData + fig.transFigure.inverted()

    if fig_specs.get("plot_pie_charts", True):
        data = None
        for experiment_instance in context.experiment_instances:
            if experiment_instance.experiment_name == experiment_name:
                data = experiment_instance.get_data()
                break
        if data is None:
            raise ValueError(f"No data found for experiment {experiment_name}")
        local_pdf_raw = context.get_best_fit_local_pdf(
            experiment_name=experiment_name,
            signal_parameter_value=plot_config["signal_parameter_value"],
            data_points=np.stack(
                (data[fig_specs["axis_names"][0]], data[fig_specs["axis_names"][1]]),
                axis=-1,
            ),
        )
        local_pdf_merged = (
            {}
        )  # Sometimes we need to merge multiple bkg into one in the pie chart
        component_color_map = {}
        best_fit_bkg_component_name = None
        signal_name = None
        for plot_item in plot_config["content_specs"]:
            if not plot_item.get("include_in_pie_charts", True):
                continue
            component_name = plot_item["component_name"]
            component_color_map[component_name] = plot_item["color"]
            if plot_item["type"] in ["bkg", "shaped_bkg"]:
                local_pdf_merged[component_name] = local_pdf_raw[component_name]
            elif plot_item["type"] == "best_fit_bkg":
                if "bkg_to_include" not in plot_item:
                    # If bkg_to_include is not given, we need to sum up all backgrounds except signal
                    # We will treat this case outside this loop
                    assert (
                        best_fit_bkg_component_name is None
                    ), "Only one best_fit_bkg without bkg_to_include is allowed"
                    best_fit_bkg_component_name = component_name
                    continue
                for bkg_name in plot_item["bkg_to_include"]:
                    if component_name in local_pdf_merged:
                        local_pdf_merged[component_name] += local_pdf_raw[bkg_name]
                    else:
                        local_pdf_merged[component_name] = local_pdf_raw[bkg_name]
            elif plot_item["type"] == "signal":
                local_pdf_merged[component_name] = local_pdf_raw[component_name]
                signal_name = component_name
        assert (
            signal_name is not None
        ), "Signal component must be included in the plot_config to draw pie charts"
        # Handle the case where best_fit_bkg includes all backgrounds except signal
        if best_fit_bkg_component_name is not None:
            total_bkg = np.zeros(len(data))
            for key in local_pdf_raw:
                if key != signal_name:
                    total_bkg += local_pdf_raw[key]
            local_pdf_merged[best_fit_bkg_component_name] = total_bkg

        data_points_pie_dtype = [
            (fig_specs["axis_names"][0], float),
            (fig_specs["axis_names"][1], float),
        ]
        for key in local_pdf_merged:
            data_points_pie_dtype.append((f"pdf_{key}", float))
        data_points_pie_dtype.append(("pie_size", float))
        data_points_pie = np.zeros(data.shape, dtype=data_points_pie_dtype)
        data_points_pie[fig_specs["axis_names"][0]] = data[fig_specs["axis_names"][0]]
        data_points_pie[fig_specs["axis_names"][1]] = data[fig_specs["axis_names"][1]]
        for key in local_pdf_merged:
            data_points_pie[f"pdf_{key}"] = local_pdf_merged[key]

        # Determine pie sizes based on signal component
        denom = np.add.reduce([local_pdf_merged[key] for key in local_pdf_merged])
        signal_relative_size = np.divide(
            local_pdf_merged[signal_name],
            denom,
            out=np.zeros_like(denom),
            where=denom > 0,
        )  # Avoid division by zero if the local pdf is all zero
        m = signal_relative_size.max()
        if m > 0:
            signal_relative_size = signal_relative_size / m
            data_points_pie["pie_size"] = fig_specs[
                "pie_size_min"
            ] + signal_relative_size * (
                fig_specs["pie_size_max"] - fig_specs["pie_size_min"]
            )
        else:
            data_points_pie["pie_size"] = fig_specs["pie_size_min"]

        # Order the pie charts by pie size (smallest drawn last)
        data_points_pie = np.sort(data_points_pie, order="pie_size")[::-1]

        for data_point_pie in data_points_pie:
            pie_center = data_to_figure.transform(
                (
                    data_point_pie[fig_specs["axis_names"][0]],
                    data_point_pie[fig_specs["axis_names"][1]],
                )
            )
            ax_pie = fig.add_axes(
                [
                    pie_center[0] - fig_specs["pie_axis_width"] / 2,
                    pie_center[1] - fig_specs["pie_axis_height"] / 2,
                    fig_specs["pie_axis_width"],
                    fig_specs["pie_axis_height"],
                ]
            )
            ax_pie.pie(
                [data_point_pie[f"pdf_{key}"] for key in local_pdf_merged],
                colors=[component_color_map[key] for key in local_pdf_merged],
                radius=data_point_pie["pie_size"],
            )

    legend_handles = [
        Patch(facecolor=plot_item["color"], alpha=1, label=plot_item["label"])
        for plot_item in plot_config["content_specs"]
    ]
    ax.legend(handles=legend_handles, **fig_specs.get("legend_kwargs", {}))
    return fig, ax

## 1. XENONnT SR0

In [ ]:
xenonnt_sr0_st = diamx.Context(
    os.path.join(diamx_dir, "..", "config", "xenonnt_sr0_wimp_config_full_roi.json"),
)
xenonnt_sr0_st.register_experiment(diamx.experiments.XENONnTSR0)
xenonnt_sr0_st.generate_templates()

In [ ]:
fig, ax = plot_contours_and_pies(
    xenonnt_sr0_st,
    {
        "experiment_name": "xenonnt_sr0",
        "signal_parameter_value": 200,
        "content_specs": [
            {
                "type": "bkg",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "component_name": "er",
                "color": "blue",
                "one_sigma_contour_fname": "xenonnt_sr0_er_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr0_er_2sigma.csv",
                "log10_contour": False,
                "label": "ER",
            },
            {
                "type": "bkg",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "component_name": "neutron",
                "color": "orange",
                "two_sigma_contour_fname": "xenonnt_sr0_neutron_2sigma.csv",
                "log10_contour": False,
                "label": "Neutron",
            },
            {
                "type": "bkg",
                "plot_contour_diamx": False,
                "plot_contour_literature": True,
                "component_name": "ac",
                "color": "purple",
                "one_sigma_contour_fname": "xenonnt_sr0_ac_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr0_ac_2sigma.csv",
                "log10_contour": False,
                "label": "AC",
            },
            {
                "type": "bkg",
                "plot_contour_diamx": False,
                "plot_contour_literature": True,
                "component_name": "surface",
                "color": "green",
                "one_sigma_contour_fname": "xenonnt_sr0_surface_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr0_surface_2sigma.csv",
                "log10_contour": False,
                "label": "Surface",
            },
            {
                "type": "signal",
                "plot_contour_diamx": False,
                "plot_contour_literature": False,
                "component_name": "wimp",
                "color": "red",
                "label": "WIMP",
            },
        ],
        "fig_specs": {
            "figure_size": [6.4, 4.8],
            "axis_names": ["cs1", "cs2"],
            "axis_labels": ["cS1 [PE]", "cS2 [PE]"],
            "axis_limits": [[0, 100], [10**2.1, 10**4.1]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 1,
            "pie_size_max": 6,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
            "legend_kwargs": {
                "loc": "upper center",
                "bbox_to_anchor": (0.5, 1.17),
                "ncol": 3,
            },
        },
    },
)
ax.axhline(400, color="black", alpha=0.8, linestyle="--")
plt.savefig("../plots/xenonnt_sr0_cs1_cs2_contours.png", dpi=600, bbox_inches="tight")

In [7]:
xenonnt_sr0_st.print_best_fit(200)

True
xenonnt_sr0
           Nominal   Best fit
er             134   135 ± 12
neutron  1.1 ± 0.5  1.1 ± 0.5
ac       4.3 ± 0.9  4.3 ± 0.8
surface     14 ± 3     12 ± 2
Signal best fit: 1.6225777732678972



In [8]:
# For a more accurate estimation of parameter uncertainties:
model = xenonnt_sr0_st.get_alea_model(200)
model.fit()
model.minuit_object.minos()

True


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1267                       │              Nfcn = 633              │
│ EDM = 2.17e-06 (Goal: 0.0001)    │            time = 0.1 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬─────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name                                │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼─────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ wimp_rate_multiplier                │  0.0022   │  0.0042   │  -0.0022   │   0.0050   │    0    │  3000   │       │
│ 1 │ mass                                │    200    │     2     │            │            │         │         │  yes  │
│ 2 │ xenonnt_sr0_livetime                │  0.2605   │  0.0026   │            │            │         │         │  yes  │
│ 3 │ xenonnt_sr0_er_rate_multiplier      │   1.01    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 4 │ xenonnt_sr0_neutron_rate_multiplier │    1.0    │    0.5    │    -0.5    │    0.5     │    0    │         │       │
│ 5 │ xenonnt_sr0_ac_rate_multiplier      │    1.0    │    0.2    │    -0.2    │    0.2     │    0    │         │       │
│ 6 │ xenonnt_sr0_surface_rate_multiplier │   0.85    │   0.17    │   -0.17    │    0.18    │    0    │         │       │
│ 7 │ xenonnt_sr0_signal_efficiency       │   1.00    │   0.05    │   -0.05    │    0.05    │   0.5   │   1.5   │       │
└───┴─────────────────────────────────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌──────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┐
│          │ wimp_rate_multiplier  │xenonnt_sr0_er_rate_multiplier│xenonnt_sr0_neutron_rate_multiplier│xenonnt_sr0_ac_rate_multiplier│xenonnt_sr0_surface_rate_multiplier│xenonnt_sr0_signal_efficiency│
├──────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┤
│  Error   │  -0.0022  │  0.0050   │   -0.09   │   0.09    │   -0.5    │    0.5    │   -0.2    │    0.2    │   -0.17   │   0.18    │   -0.05   │   0.05    │
│  Valid   │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │
│ At Limit │   True    │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │
│ Max FCN  │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │
│ New Min  │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │
└──────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┘
┌─────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# Reproducing Fig. 4 (upper limit)
# xenonnt_sr0_st.run_inference(output_file_name="xenonnt_sr0_wimp_ci_full_roi.csv")
diamx.run_inference_pool(
    xenonnt_sr0_st,
    output_file_name="xenonnt_sr0_wimp_ci_full_roi.csv",
    processes=8,
)
ci = np.loadtxt("./diamx_output/xenonnt_sr0_wimp_ci_full_roi.csv", delimiter=",")
plt.plot(ci[:, 0], ci[:, 2] * 1e-44, label="diamx")
plt.yscale("log")
plt.xscale("log")
ci_literature = np.loadtxt("../diamx/data/xenonnt_sr0_wimp_ci.csv", delimiter=",")
plt.plot(ci_literature[:, 0], ci_literature[:, 1], label="literature")
plt.legend()
plt.yscale("log")
plt.xscale("log")
plt.xlabel("WIMP mass [GeV]")
plt.ylabel("Cross section [cm^2]")
plt.title("XENONnT SR0 WIMP 90% CI")
plt.savefig("../plots/xenonnt_sr0_wimp_ci_full_roi.png", dpi=600)

Alternatively, use `cS2 > 400PE`:

In [3]:
xenonnt_sr0_st_400PE = diamx.Context(
    os.path.join(diamx_dir, "..", "config", "xenonnt_sr0_wimp_config.json"),
)
xenonnt_sr0_st_400PE.register_experiment(diamx.experiments.XENONnTSR0)
xenonnt_sr0_st_400PE.generate_templates()
xenonnt_sr0_st_400PE.print_best_fit(200)

Generating background templates for xenonnt_sr0: 100%|██████████| 4/4 [00:00<00:00, 1348.98it/s]
Generating shaped background templates for xenonnt_sr0: 0it [00:00, ?it/s]
Generating signal templates for xenonnt_sr0: 100%|██████████| 90/90 [00:00<00:00, 1826.50it/s]


True
xenonnt_sr0
           Nominal   Best fit
er             134   134 ± 12
neutron  1.1 ± 0.5  1.1 ± 0.5
ac               1  0.0 ± 0.7
surface          1  3.0 ± 1.7
Signal best fit: 3.164179930384791



In [4]:
alea_model_400PE = xenonnt_sr0_st_400PE.get_alea_model(200)
alea_model_400PE.fit()
alea_model_400PE.minuit_object.minos()

True


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1184                       │              Nfcn = 780              │
│ EDM = 1.22e-05 (Goal: 0.0001)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬─────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name                                │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼─────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ wimp_rate_multiplier                │   0.004   │   0.005   │   -0.004   │   0.006    │    0    │  3000   │       │
│ 1 │ mass                                │    200    │     2     │            │            │         │         │  yes  │
│ 2 │ xenonnt_sr0_livetime                │  0.2605   │  0.0026   │            │            │         │         │  yes  │
│ 3 │ xenonnt_sr0_er_rate_multiplier      │   1.00    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 4 │ xenonnt_sr0_neutron_rate_multiplier │    1.0    │    0.5    │    -0.5    │    0.5     │    0    │         │       │
│ 5 │ xenonnt_sr0_ac_rate_multiplier      │  0.24e-6  │718799.11e-6│  -0.23e-6  │965699.59e-6│    0    │         │       │
│ 6 │ xenonnt_sr0_surface_rate_multiplier │    3.0    │    1.7    │    -1.4    │    2.1     │    0    │         │       │
│ 7 │ xenonnt_sr0_signal_efficiency       │   1.00    │   0.05    │   -0.05    │    0.05    │   0.5   │   1.5   │       │
└───┴─────────────────────────────────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌──────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┐
│          │ wimp_rate_multiplier  │xenonnt_sr0_er_rate_multiplier│xenonnt_sr0_neutron_rate_multiplier│xenonnt_sr0_ac_rate_multiplier│xenonnt_sr0_surface_rate_multiplier│xenonnt_sr0_signal_efficiency│
├──────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┤
│  Error   │  -0.004   │   0.006   │   -0.09   │   0.09    │   -0.5    │    0.5    │ -0.23e-6  │965699.59e-6│   -1.4    │    2.1    │   -0.05   │   0.05    │
│  Valid   │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │
│ At Limit │   False   │   False   │   False   │   False   │   False   │   False   │   True    │   False   │   False   │   False   │   False   │   False   │
│ Max FCN  │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │
│ New Min  │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │
└──────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┴───────────┘
┌─────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# xenonnt_sr0_st_400PE.run_inference(output_file_name="xenonnt_sr0_wimp_ci_400PE.csv")
diamx.run_inference_pool(
    xenonnt_sr0_st_400PE,
    output_file_name="xenonnt_sr0_wimp_ci_400PE.csv",
    processes=8,
)
ci_400PE = np.loadtxt("./diamx_output/xenonnt_sr0_wimp_ci_400PE.csv", delimiter=",")
plt.plot(ci_400PE[:, 0], ci_400PE[:, 2] * 1e-44, label="diamx (cS2 < 400PE)")
ci_literature = np.loadtxt("../diamx/data/xenonnt_sr0_wimp_ci.csv", delimiter=",")
plt.plot(ci_literature[:, 0], ci_literature[:, 1], label="literature")
plt.legend()
plt.yscale("log")
plt.xscale("log")
plt.xlabel("WIMP mass [GeV]")
plt.ylabel("Cross section [cm^2]")
plt.title("XENONnT SR0 WIMP 90% CI")
plt.savefig("../plots/xenonnt_sr0_wimp_ci_400PE.png", dpi=600, bbox_inches="tight")

## 2. LZ WS2022

In [ ]:
lz_ws2022_st = diamx.Context(
    os.path.join(diamx_dir, "..", "config", "lz_ws2022_wimp_config.json"),
)
lz_ws2022_st.register_experiment(diamx.experiments.LZWS2022)
# lz_ws2022_st.prepare()
lz_ws2022_st.generate_templates()

In [ ]:
plot_contours_and_pies(
    lz_ws2022_st,
    {
        "experiment_name": "lz_ws2022",
        "signal_parameter_value": 30,
        "content_specs": [
            {
                "type": "best_fit_bkg",
                "component_name": "er",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "color": "mediumblue",
                "one_sigma_contour_fname": "lz_ws2022_bkg_1sigma.csv",
                "two_sigma_contour_fname": "lz_ws2022_bkg_2sigma.csv",
                "log10_contour": True,
                "bkg_to_include": [
                    "beta",
                    "neutrino",
                    "xe127",
                    "xe124",
                    "xe136",
                    "ar37",
                ],
                "label": "ER",
            },
            {
                "type": "bkg",
                "component_name": "ar37",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "include_in_pie_charts": False,
                "color": "green",
                "one_sigma_contour_fname": "lz_ws2022_ar37_1sigma.csv",
                "two_sigma_contour_fname": "lz_ws2022_ar37_2sigma.csv",
                "log10_contour": True,
                "label": "Ar37",
            },
            {
                "type": "signal",
                "component_name": "wimp",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "color": "red",
                "one_sigma_contour_fname": "lz_ws2022_wimp_1sigma.csv",
                "two_sigma_contour_fname": "lz_ws2022_wimp_2sigma.csv",
                "log10_contour": True,
                "label": "WIMP",
            },
        ],
        "fig_specs": {
            "figure_size": [6.4, 4.8],
            "axis_names": ["s1c", "s2c"],
            "axis_labels": ["S1c [phd]", "S2c [phd]"],
            "axis_limits": [[3, 80], [600, 10**4.5]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 1,
            "pie_size_max": 1,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
            "legend_kwargs": {
                "loc": "upper center",
                "bbox_to_anchor": (0.5, 1.1),
                "ncol": 3,
            },
        },
    },
)

plt.savefig("../plots/lz_ws2022_s1c_s2c_contours.png", dpi=600, bbox_inches="tight")

In [13]:
lz_ws2022_st.print_best_fit(30, disable_rounding=True, stabilize_fit=False)

True
lz_ws2022
                           Nominal                                 Best fit
beta                  215.0 ± 36.0  223.96491535603718 ± 16.169763457608962
neutrino                27.1 ± 1.6   27.11838691279395 ± 1.5984846828021946
xe127                    9.2 ± 0.8   9.263876047153692 ± 0.7931419607130025
xe124     5.0 ± 1.3999999999999997   5.322142846393244 ± 1.3765318410698246
xe136                   15.1 ± 2.4   15.162610792510895 ± 2.389411149628825
ar37                          50.0     50.48584296372327 ± 9.08578730862105
Signal best fit: 5.66448843555421e-06



In [14]:
model = lz_ws2022_st.get_alea_model(30)
model.fit()
model.minuit_object.minos()

True


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 2820                       │             Nfcn = 1053              │
│ EDM = 8.57e-06 (Goal: 0.0001)    │            time = 0.3 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name                               │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ wimp_rate_multiplier               │   4e-9    │ 463449e-9 │   -4e-9    │ 463058e-9  │    0    │  3000   │       │
│ 1 │ mass                               │   30.0    │    0.3    │            │            │         │         │  yes  │
│ 2 │ lz_ws2022_livetime                 │  0.1644   │  0.0016   │            │            │         │         │  yes  │
│ 3 │ lz_ws2022_beta_rate_multiplier     │   1.04    │   0.08    │   -0.07    │    0.08    │    0    │         │       │
│ 4 │ lz_ws2022_neutrino_rate_multiplier │   1.00    │   0.06    │   -0.06    │    0.06    │    0    │         │       │
│ 5 │ lz_ws2022_xe127_rate_multiplier    │   1.01    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 6 │ lz_ws2022_xe124_rate_multiplier    │   1.06    │   0.28    │   -0.28    │    0.28    │    0    │         │       │
│ 7 │ lz_ws2022_xe136_rate_multiplier    │   1.00    │   0.16    │   -0.16    │    0.16    │    0    │         │       │
│ 8 │ lz_ws2022_ar37_rate_multiplier     │   1.01    │   0.18    │   -0.18    │    0.19    │    0    │  5.76   │       │
│ 9 │ lz_ws2022_signal_efficiency        │   1.000   │   0.025   │   -0.025   │   0.025    │   0.5   │   1.5   │       │
└───┴────────────────────────────────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌──────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┐
│          │ wimp_rate_multiplier  │lz_ws2022_beta_rate_multiplier│lz_ws2022_neutrino_rate_multiplier│lz_ws2022_xe127_rate_multiplier│lz_ws2022_xe124_rate_multiplier│lz_ws2022_xe136_rate_multiplier│lz_ws2022_ar37_rate_multiplier│lz_ws2022_signal_efficiency│
├──────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┼───────────┬───────────┤
│  Error   │   -4e-9   │ 463058e-9 │   -0.07   │   0.08    │   -0.06   │   0.06    │   -0.09   │   0.09    │   -0.28   │   0.28    │   -0.16   │   0.16    │   -0.18   │   0.19    │  -0.025   │   0.025   │
│  Valid   │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │   True    │
│ At Limit │   True    │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │
│ Max FCN  │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   False   │   

In [ ]:
# Reproducing Fig. 5 (upper limit)
# lz_ws2022_st.run_inference(output_file_name="lz_ws2022_wimp_ci.csv")
diamx.run_inference_pool(
    lz_ws2022_st,
    output_file_name="lz_ws2022_wimp_ci.csv",
    processes=8,
)
ci = np.loadtxt("./diamx_output/lz_ws2022_wimp_ci.csv", delimiter=",")
plt.plot(ci[:, 0], ci[:, 2] * 1e-44, label="diamx")
plt.yscale("log")
plt.xscale("log")
ci_literature = np.loadtxt("../diamx/data/lz_ws2022_wimp_ci.csv", delimiter=",")
plt.plot(ci_literature[:, 0], ci_literature[:, 1], label="literature")
plt.legend()
plt.yscale("log")
plt.xscale("log")
plt.xlabel("WIMP mass [GeV]")
plt.ylabel("Cross section [cm^2]")
plt.title("LZ WS2022 WIMP 90% CI")
plt.savefig("../plots/lz_ws2022_wimp_ci.png", dpi=600)

## 3. XENONnT combined SR0 & SR1

ROI: cS2 > 400PE

In [ ]:
xenonnt_sr01_st = diamx.Context(
    os.path.join(diamx_dir, "..", "config", "xenonnt_sr0_and_1_wimp_config.json"),
)
xenonnt_sr01_st.register_experiment(diamx.experiments.XENONnTSR0)
xenonnt_sr01_st.register_experiment(diamx.experiments.XENONnTSR1a)
xenonnt_sr01_st.register_experiment(diamx.experiments.XENONnTSR1b)
# xenonnt_sr01_st.prepare()
xenonnt_sr01_st.generate_templates()

In [ ]:
plot_contours_and_pies(
    xenonnt_sr01_st,
    {
        "experiment_name": "xenonnt_sr1a",
        "signal_parameter_value": 200,
        "content_specs": [
            {
                "type": "best_fit_bkg",
                "component_name": "er",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "bkg_to_include": ["er_flat", "tritium", "ar37"],
                "color": "blue",
                "one_sigma_contour_fname": "xenonnt_sr1a_er_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1a_er_2sigma.csv",
                "log10_contour": False,
                "label": "ER",
            },
            {
                "type": "signal",
                "component_name": "wimp",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "color": "red",
                "one_sigma_contour_fname": "xenonnt_sr1b_wimp_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_wimp_2sigma.csv",
                "log10_contour": False,
                "label": "WIMP",
            },
            {
                "type": "bkg",
                "component_name": "xenonnt_sr1_ac",
                "plot_contour_diamx": False,
                "plot_contour_literature": True,
                "color": "purple",
                "one_sigma_contour_fname": "xenonnt_sr1b_ac_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_ac_2sigma.csv",
                "log10_contour": False,
                "label": "AC",
            },
        ],
        "fig_specs": {
            "figure_size": [6.4, 4.8],
            "axis_names": ["cs1", "cs2"],
            "axis_labels": ["cS1 [PE]", "cS2 [PE]"],
            "axis_limits": [[0, 100], [10**2.1, 10**4.1]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 1,
            "pie_size_max": 6,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
            "legend_kwargs": {
                "loc": "upper center",
                "bbox_to_anchor": (0.5, 1.1),
                "ncol": 4,
            },
        },
    },
)
plt.axhline(400, color="k", linestyle="--", alpha=0.2)
plt.savefig("../plots/xenonnt_sr1a_cs1_cs2_contours.png", dpi=600, bbox_inches="tight")

In [ ]:
plot_contours_and_pies(
    xenonnt_sr01_st,
    {
        "experiment_name": "xenonnt_sr1b",
        "signal_parameter_value": 200,
        "content_specs": [
            {
                "type": "best_fit_bkg",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "component_name": "er",
                "bkg_to_include": ["er_flat", "tritium"],
                "color": "blue",
                "one_sigma_contour_fname": "xenonnt_sr1b_er_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_er_2sigma.csv",
                "log10_contour": False,
                "label": "ER",
            },
            {
                "type": "signal",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "component_name": "wimp",
                "color": "red",
                "one_sigma_contour_fname": "xenonnt_sr1b_wimp_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_wimp_2sigma.csv",
                "log10_contour": False,
                "label": "WIMP",
            },
            {
                "type": "bkg",
                "plot_contour_diamx": False,
                "plot_contour_literature": True,
                "component_name": "xenonnt_sr1_ac",
                "color": "purple",
                "one_sigma_contour_fname": "xenonnt_sr1b_ac_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_ac_2sigma.csv",
                "log10_contour": False,
                "label": "AC",
            },
        ],
        "fig_specs": {
            "figure_size": [6.4, 4.8],
            "axis_names": ["cs1", "cs2"],
            "axis_labels": ["cS1 [PE]", "cS2 [PE]"],
            "axis_limits": [[0, 100], [10**2.1, 10**4.1]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 1,
            "pie_size_max": 6,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
            "legend_kwargs": {
                "loc": "upper center",
                "bbox_to_anchor": (0.5, 1.1),
                "ncol": 4,
            },
        },
    },
)

plt.axhline(400, color="k", linestyle="--", alpha=0.2)
plt.savefig("../plots/xenonnt_sr1b_cs1_cs2_contours.png", dpi=600, bbox_inches="tight")

In [20]:
xenonnt_sr01_st.print_best_fit(200)

True
True
True
xenonnt_sr0
                   Nominal   Best fit
er                     134   135 ± 12
xenonnt_neutron  0.7 ± 0.3  0.7 ± 0.3
ac                       1  0.0 ± 0.7
surface                  1  3.0 ± 1.7
Signal best fit: 1.8175673023721037

xenonnt_sr1a
                     Nominal     Best fit
er_flat             430 ± 30     460 ± 30
tritium                   62      20 ± 30
ar37                  58 ± 6       53 ± 5
xenonnt_neutron  0.47 ± 0.19  0.47 ± 0.19
xenonnt_sr1_ac   2.12 ± 0.18  2.11 ± 0.18
Signal best fit: 1.2180578724785507

xenonnt_sr1b
                   Nominal   Best fit
er_flat           151 ± 11   157 ± 10
tritium                101    69 ± 17
xenonnt_neutron  0.7 ± 0.3  0.7 ± 0.3
xenonnt_sr1_ac   3.8 ± 0.3  3.8 ± 0.3
Signal best fit: 2.19286995360628



In [21]:
model = xenonnt_sr01_st.get_alea_model(200)
model.fit()
model.minuit_object.minos()

True
True
True


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 6707                       │             Nfcn = 3277              │
│ EDM = 1.76e-06 (Goal: 0.0001)    │            time = 2.0 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name                                 │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ wimp_rate_multiplier                 │  0.0025   │  0.0026   │  -0.0023   │   0.0029   │    0    │  3000   │       │
│ 1 │ mass                                 │    200    │     2     │            │            │         │         │  yes  │
│ 2 │ xenonnt_sr0_livetime                 │  0.2605   │  0.0026   │            │            │         │         │  yes  │
│ 3 │ xenonnt_sr0_er_rate_multiplier       │   1.01    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 4 │ xenonnt_neutron_rate_multiplier      │    1.0    │    0.4    │    -0.4    │    0.4     │    0    │         │       │
│ 5 │ xenonnt_sr0_ac_rate_multiplier       │ 0.031e-6  │748802.589e-6│ -0.032e-6  │1012862.140e-6│    0    │         │       │
│ 6 │ xenonnt_sr0_surface_rate_multiplier  │    3.0    │    1.7    │    -1.4    │    2.1     │    0    │         │       │
│ 7 │ xenonnt_sr0_signal_efficiency        │   1.00    │   0.05    │   -0.05    │    0.05    │   0.5   │   1.5   │       │
│ 8 │ xenonnt_sr1a_livetime                │  0.1825   │  0.0018   │            │            │         │         │  yes  │
│ 9 │ xenonnt_sr1a_er_flat_rate_multiplier │   1.06    │   0.06    │   -0.06    │    0.06    │    0    │         │       │
│ 10│ xenonnt_sr1a_tritium_rate_multiplier │   0.35    │   0.48    │   -0.35    │    0.53    │    0    │         │       │
│ 11│ xenonnt_sr1a_ar37_rate_multiplier    │   0.91    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 12│ xenonnt_sr1_ac_rate_multiplier       │   1.00    │   0.08    │   -0.08    │    0.08    │    0    │         │       │
│ 13│ xenonnt_sr1a_signal_efficiency       │   1.00    │   0.04    │   -0.04    │    0.04    │   0.5   │   1.5   │       │
│ 14│ xenonnt_sr1b_livetime                │  0.3285   │  0.0033   │            │            │         │         │  yes  │
│ 15│ xenonnt_sr1b_er_flat_rate_multiplier │   1.04    │   0.07    │   -0.07    │    0.07    │    0    │         │       │
│ 16│ xenonnt_sr1b_tritium_rate_multiplier │   0.69    │   0.17    │   -0.17    │    0.18    │    0    │         │       │
│ 17│ xenonnt_sr1b_signal_efficiency       │   1.00    │   0.04    │   -0.04    │    0.04    │   0.5   │   1.5   │       │
└───┴──────────────────────────────────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌──────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┐
│          │ wimp_rate_multip

In [ ]:
# xenonnt_sr01_st.run_inference(output_file_name="xenonnt_sr1_wimp_ci_full_roi.csv")
diamx.run_inference_pool(
    xenonnt_sr01_st,
    output_file_name="xenonnt_sr1_wimp_ci.csv",
    processes=8,
)
ci = np.loadtxt("./diamx_output/xenonnt_sr1_wimp_ci.csv", delimiter=",")
plt.plot(ci[:, 0], ci[:, 2] * 1e-44, label="diamx")
ci_literature = np.loadtxt("../diamx/data/xenonnt_sr1_wimp_ci.csv", delimiter=",")
plt.plot(ci_literature[:, 0], ci_literature[:, 1], label="literature")
plt.legend()
plt.yscale("log")
plt.xscale("log")
plt.xlabel("WIMP mass [GeV]")
plt.ylabel("Cross section [cm^2]")
plt.title("XENONnT SR0&1 WIMP 90% CI")
plt.savefig("../plots/xenonnt_sr1_wimp_ci.png", dpi=600)
# plt.show()

Alternatively, introducing two floating parameters for Xe124 DEC charge yield:

In [ ]:
xenonnt_sr01_dec_st = diamx.Context(
    os.path.join(diamx_dir, "..", "config", "xenonnt_sr0_and_1_wimp_config_dec.json"),
)
xenonnt_sr01_dec_st.register_experiment(diamx.experiments.XENONnTSR0)
xenonnt_sr01_dec_st.register_experiment(diamx.experiments.XENONnTSR1a)
xenonnt_sr01_dec_st.register_experiment(diamx.experiments.XENONnTSR1b)
# xenonnt_sr01_dec_st.prepare()
xenonnt_sr01_dec_st.generate_templates()

In [ ]:
diamx.run_inference_pool(
    xenonnt_sr01_dec_st,
    processes=10,
    output_file_name="xenonnt_sr01_dec_wimp_ci.csv",
    maxtasksperchild=1,
)

In [ ]:
ci_dec = np.loadtxt("./diamx_output/xenonnt_sr01_dec_wimp_ci.csv", delimiter=",")
plt.plot(ci_dec[:, 0], ci_dec[:, 2] * 1e-44, label="diamx (free DEC)")
ci_literature = np.loadtxt("../diamx/data/xenonnt_sr1_dec_wimp_ci.csv", delimiter=",")
plt.plot(ci_literature[:, 0], ci_literature[:, 1], label="literature")
plt.xlabel("WIMP mass [GeV]")
plt.ylabel("Cross section [cm$^2$]")
plt.yscale("log")
plt.xscale("log")
plt.legend()
plt.savefig("../plots/xenonnt_sr1_dec_wimp_ci.png", dpi=600, bbox_inches="tight")

In [ ]:
plot_contours_and_pies(
    xenonnt_sr01_dec_st,
    {
        "experiment_name": "xenonnt_sr1b",
        "signal_parameter_value": 200,
        "content_specs": [
            {
                "type": "best_fit_bkg",
                "plot_contour_diamx": True,
                "plot_contour_literature": True,
                "component_name": "er",
                "bkg_to_include": ["er_flat", "tritium"],
                "color": "blue",
                "one_sigma_contour_fname": "xenonnt_sr1b_er_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_er_2sigma.csv",
                "log10_contour": False,
                "label": "ER",
            },
            {
                "type": "signal",
                "plot_contour_diamx": True,
                "plot_contour_literature": True,
                "component_name": "wimp",
                "color": "red",
                "one_sigma_contour_fname": "xenonnt_sr1b_wimp_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_wimp_2sigma.csv",
                "log10_contour": False,
                "label": "WIMP",
            },
            {
                "type": "bkg",
                "plot_contour_diamx": False,
                "plot_contour_literature": True,
                "component_name": "xenonnt_sr1_ac",
                "color": "purple",
                "one_sigma_contour_fname": "xenonnt_sr1b_ac_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_ac_2sigma.csv",
                "log10_contour": False,
                "label": "AC",
            },
            {
                "type": "best_fit_bkg",
                "plot_contour_diamx": True,
                "plot_contour_literature": False,
                "component_name": "xenonnt_sr1_xe124",
                "bkg_to_include": ["xenonnt_sr1_xe124"],
                "color": "indigo",
                "label": "Xe124",
            },
        ],
        "fig_specs": {
            "figure_size": [6.4, 4.8],
            "axis_names": ["cs1", "cs2"],
            "axis_labels": ["cS1 [PE]", "cS2 [PE]"],
            "axis_limits": [[0, 100], [10**2.1, 10**4.1]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 1,
            "pie_size_max": 6,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
            "legend_kwargs": {
                "loc": "upper center",
                "bbox_to_anchor": (0.5, 1.1),
                "ncol": 4,
            },
        },
    },
)

In [ ]:
plot_contours_and_pies(
    xenonnt_sr01_dec_st,
    {
        "experiment_name": "xenonnt_sr1b",
        "signal_parameter_value": 200,
        "content_specs": [
            {
                "type": "best_fit_bkg",
                "plot_contour_diamx": True,
                "plot_contour_literature": True,
                "component_name": "er",
                "bkg_to_include": ["er_flat", "tritium"],
                "color": "blue",
                "one_sigma_contour_fname": "xenonnt_sr1b_er_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_er_2sigma.csv",
                "log10_contour": False,
                "label": "ER",
            },
            {
                "type": "signal",
                "plot_contour_diamx": True,
                "plot_contour_literature": True,
                "component_name": "wimp",
                "color": "red",
                "one_sigma_contour_fname": "xenonnt_sr1b_wimp_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_wimp_2sigma.csv",
                "log10_contour": False,
                "label": "WIMP",
            },
            {
                "type": "bkg",
                "plot_contour_diamx": False,
                "plot_contour_literature": True,
                "component_name": "xenonnt_sr1_ac",
                "color": "purple",
                "one_sigma_contour_fname": "xenonnt_sr1b_ac_1sigma.csv",
                "two_sigma_contour_fname": "xenonnt_sr1b_ac_2sigma.csv",
                "log10_contour": False,
                "label": "AC",
            },
            {
                "type": "shaped_bkg",
                "plot_contour_diamx": True,
                "plot_contour_literature": False,
                "component_name": "xenonnt_sr1_xe124",
                "shape_parameter_value": (1.00, 1.00),
                "color": "indigo",
                "label": "Xe124",
            },
        ],
        "fig_specs": {
            "figure_size": [6.4, 4.8],
            "axis_names": ["cs1", "cs2"],
            "axis_labels": ["cS1 [PE]", "cS2 [PE]"],
            "axis_limits": [[0, 100], [10**2.1, 10**4.1]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 1,
            "pie_size_max": 6,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
            "legend_kwargs": {
                "loc": "upper center",
                "bbox_to_anchor": (0.5, 1.1),
                "ncol": 4,
            },
        },
    },
)

In [10]:
xenonnt_sr01_dec_st.print_best_fit(200)

True
True
True
xenonnt_sr0
                   Nominal   Best fit
er                     130   126 ± 11
xenonnt_neutron  0.7 ± 0.3  0.7 ± 0.3
ac                       1  0.0 ± 0.8
surface                  1  3.0 ± 1.7
xe124            4.5 ± 0.7  4.7 ± 0.7
Shape parameters:
                    Nominal       Best fit
ll_quenching_factor       1  0.820 ± 0.017
lm_quenching_factor       1  0.740 ± 0.008
Signal best fit: 0.6503648069066161

xenonnt_sr1a
                       Nominal     Best fit
er_flat               430 ± 30     450 ± 30
tritium                     62      20 ± 30
ar37                    58 ± 6       53 ± 5
xenonnt_neutron    0.47 ± 0.19  0.47 ± 0.18
xenonnt_sr1_ac     2.12 ± 0.18  2.12 ± 0.18
xenonnt_sr1_xe124    3.2 ± 0.5    3.3 ± 0.5
Shape parameters:
                    Nominal       Best fit
ll_quenching_factor       1  0.820 ± 0.017
lm_quenching_factor       1  0.740 ± 0.008
Signal best fit: 0.43584739448257065

xenonnt_sr1b
                     Nominal   Best fit
er

In [11]:
alea_model = xenonnt_sr01_dec_st.get_alea_model(200)
alea_model.fit()
alea_model.minuit_object.minos()

True
True
True


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 6701                       │             Nfcn = 7266              │
│ EDM = 0.00024 (Goal: 0.0001)     │            time = 6.9 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name                                 │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ wimp_rate_multiplier                 │  0.0012   │  0.0021   │  -0.0012   │   0.0025   │    0    │  3000   │       │
│ 1 │ mass                                 │    200    │     2     │            │            │         │         │  yes  │
│ 2 │ xenonnt_sr0_livetime                 │  0.2605   │  0.0026   │            │            │         │         │  yes  │
│ 3 │ xenonnt_sr0_er_rate_multiplier       │   0.97    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 4 │ xenonnt_neutron_rate_multiplier      │    1.0    │    0.4    │    -0.4    │    0.4     │    0    │         │       │
│ 5 │ xenonnt_sr0_ac_rate_multiplier       │   5e-6    │ 801487e-6 │   -5e-6    │ 1098398e-6 │    0    │         │       │
│ 6 │ xenonnt_sr0_surface_rate_multiplier  │    3.0    │    1.7    │    -1.4    │    2.1     │    0    │         │       │
│ 7 │ xenonnt_sr0_xe124_rate_multiplier    │   1.04    │   0.15    │   -0.15    │    0.15    │    0    │         │       │
│ 8 │ ll_quenching_factor                  │   0.820   │   0.018   │   -0.078   │   0.080    │   0.6   │   0.9   │       │
│ 9 │ lm_quenching_factor                  │   0.740   │   0.015   │   -0.061   │   0.072    │   0.6   │   0.9   │       │
│ 10│ xenonnt_sr0_signal_efficiency        │   1.00    │   0.05    │   -0.05    │    0.05    │   0.5   │   1.5   │       │
│ 11│ xenonnt_sr1a_livetime                │  0.1825   │  0.0018   │            │            │         │         │  yes  │
│ 12│ xenonnt_sr1a_er_flat_rate_multiplier │   1.06    │   0.06    │   -0.06    │    0.06    │    0    │         │       │
│ 13│ xenonnt_sr1a_tritium_rate_multiplier │   0.31    │   0.47    │   -0.31    │    0.53    │    0    │         │       │
│ 14│ xenonnt_sr1a_ar37_rate_multiplier    │   0.91    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 15│ xenonnt_sr1_ac_rate_multiplier       │   1.00    │   0.08    │   -0.08    │    0.08    │    0    │         │       │
│ 16│ xenonnt_sr1_xe124_rate_multiplier    │   1.00    │   0.15    │   -0.15    │    0.15    │    0    │         │       │
│ 17│ xenonnt_sr1a_signal_efficiency       │   1.00    │   0.04    │   -0.04    │    0.04    │   0.5   │   1.5   │       │
│ 18│ xenonnt_sr1b_livetime                │  0.3285   │  0.0033   │            │            │         │         │  yes  │
│ 19│ xenonnt_sr1b_er_flat_rate_multiplier │   1.03    │   0.07    │   -0.07    │    0.07    │    0    │         │       │
│ 20│ xenonnt_sr1b_tritium_rate_multiplier │   0.65    │   0.17    │   -0.17    │    0.18    │    0    │         │       │
│ 21│ xenonnt_sr1b_signal_efficiency       │   1.00    │   0.04    │   -0.04    │    0.04    │   0.5   │   1.5   │       │
└───┴────────

## 4. LZ WS2024

In [ ]:
lz_ws2024_st = diamx.Context(
    os.path.join(diamx_dir, "..", "config", "lz_ws2024_wimp_config.json"),
)
lz_ws2024_st.register_experiment(diamx.experiments.LZWS2022)
lz_ws2024_st.register_experiment(diamx.experiments.LZWS2024)
lz_ws2024_st.generate_templates()
# lz_ws2024_st.prepare()

In [ ]:
plot_contours_and_pies(
    lz_ws2024_st,
    {
        "experiment_name": "lz_ws2024",
        "signal_parameter_value": 40,
        "content_specs": [
            {
                "type": "best_fit_bkg",
                "component_name": "er",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "bkg_to_include": [
                    "pb214",
                    "kr85_ar39_detgamma",
                    "solar_neutrino_er",
                    "pb212_po218",
                    "tritium_c14",
                    "xe136",
                    "xe127_xe125",
                ],
                "color": "blue",
                "one_sigma_contour_fname": "lz_ws2024_er_1sigma.csv",
                "two_sigma_contour_fname": "lz_ws2024_er_2sigma.csv",
                "log10_contour": True,
                "label": "ER",
            },
            {
                "type": "signal",
                "component_name": "wimp",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "color": "red",
                "one_sigma_contour_fname": "lz_ws2024_wimp_1sigma.csv",
                "two_sigma_contour_fname": "lz_ws2024_wimp_2sigma.csv",
                "log10_contour": True,
                "label": "WIMP",
            },
            {
                "type": "bkg",
                "component_name": "ac",
                "plot_contour_diamx": False,
                "plot_contour_literature": True,
                "color": "purple",
                "one_sigma_contour_fname": "lz_ws2024_ac_1sigma.csv",
                "two_sigma_contour_fname": "lz_ws2024_ac_2sigma.csv",
                "log10_contour": True,
                "label": "AC",
            },
            {
                "type": "shaped_bkg",
                "component_name": "xe124",
                "plot_contour_diamx": "contour",
                "plot_contour_literature": True,
                "color": "k",
                "shape_parameter_value": [0.69],
                "one_sigma_contour_fname": [
                    "lz_ws2024_xe124_1sigma_left.csv",
                    "lz_ws2024_xe124_1sigma_right.csv",
                ],
                "two_sigma_contour_fname": "lz_ws2024_xe124_2sigma.csv",
                "log10_contour": True,
                "label": r"Xe124",
            },
        ],
        "fig_specs": {
            "figure_size": [6.4, 4.8],
            "axis_names": ["s1c", "s2c"],
            "axis_labels": ["S1c [phd]", "S2c [phd]"],
            "axis_limits": [[3, 80], [645, 10**4.5]],
            "axis_scales": ["linear", "log"],
            "contour_linestyles": ["--", "-"],
            "plot_pie_charts": True,
            "pie_size_min": 1,
            "pie_size_max": 1,
            "pie_axis_width": 0.01,
            "pie_axis_height": 0.01,
            "legend_kwargs": {
                "loc": "upper center",
                "bbox_to_anchor": (0.5, 1.1),
                "ncol": 4,
            },
        },
    },
)
plt.savefig("../plots/lz_ws2024_s1c_s2c_contours.png", dpi=600, bbox_inches="tight")

In [18]:
lz_ws2024_st.print_best_fit(40, disable_rounding=True, stabilize_fit=False)

True
True
lz_ws2022
                           Nominal                                Best fit
beta                  215.0 ± 36.0   224.0114838190287 ± 16.16842697211983
neutrino                27.1 ± 1.6  27.11755933655485 ± 1.5984691307729146
xe127                    9.2 ± 0.8    9.2635940910044 ± 0.7931432055249612
xe124     5.0 ± 1.3999999999999997  5.321038219233181 ± 1.3765479103462497
xe136                   15.1 ± 2.4   15.15988060445218 ± 2.389375466231542
ar37                          50.0   50.46838511834779 ± 9.083617490725992
Signal best fit: 1.9409655077200962e-06

lz_ws2024
                                    Nominal  \
pb214                          743.0 ± 88.0   
kr85_ar39_detgamma             162.0 ± 22.0   
solar_neutrino_er               102.0 ± 6.0   
pb212_po218                      62.7 ± 7.5   
tritium_c14         58.29999999999999 ± 3.3   
xe136                            55.6 ± 8.3   
xe127_xe125                       3.2 ± 0.6   
ac                          

In [19]:
alea_model = lz_ws2024_st.get_alea_model(40)
alea_model.fit()
alea_model.minuit_object.minos()

True
True


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.114e+04                  │             Nfcn = 7788              │
│ EDM = 4e-05 (Goal: 0.0001)       │           time = 10.3 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────────────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name                                         │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────────────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ wimp_rate_multiplier                         │  1.1e-9   │101031.4e-9│  -1.1e-9   │ 99593.3e-9 │    0    │  3000   │       │
│ 1 │ mass                                         │   40.0    │    0.4    │            │            │         │         │  yes  │
│ 2 │ lz_ws2022_livetime                           │  0.1644   │  0.0016   │            │            │         │         │  yes  │
│ 3 │ lz_ws2022_beta_rate_multiplier               │   1.04    │   0.08    │   -0.07    │    0.08    │    0    │         │       │
│ 4 │ lz_ws2022_neutrino_rate_multiplier           │   1.00    │   0.06    │   -0.06    │    0.06    │    0    │         │       │
│ 5 │ lz_ws2022_xe127_rate_multiplier              │   1.01    │   0.09    │   -0.09    │    0.09    │    0    │         │       │
│ 6 │ lz_ws2022_xe124_rate_multiplier              │   1.06    │   0.28    │   -0.28    │    0.28    │    0    │         │       │
│ 7 │ lz_ws2022_xe136_rate_multiplier              │   1.00    │   0.16    │   -0.16    │    0.16    │    0    │         │       │
│ 8 │ lz_ws2022_ar37_rate_multiplier               │   1.01    │   0.18    │   -0.17    │    0.19    │    0    │  5.76   │       │
│ 9 │ lz_ws2022_signal_efficiency                  │   1.000   │   0.024   │   -0.024   │   0.024    │   0.5   │   1.5   │       │
│ 10│ lz_ws2024_livetime                           │   0.603   │   0.006   │            │            │         │         │  yes  │
│ 11│ lz_ws2024_pb214_rate_multiplier              │   1.01    │   0.05    │   -0.05    │    0.05    │    0    │         │       │
│ 12│ lz_ws2024_kr85_ar39_detgamma_rate_multiplier │   1.00    │   0.13    │   -0.13    │    0.13    │    0    │         │       │
│ 13│ lz_ws2024_solar_neutrino_er_rate_multiplier  │   1.00    │   0.06    │   -0.06    │    0.06    │    0    │         │       │
│ 14│ lz_ws2024_pb212_po218_rate_multiplier        │   1.00    │   0.12    │   -0.12    │    0.12    │    0    │         │       │
│ 15│ lz_ws2024_tritium_c14_rate_multiplier        │   1.00    │   0.06    │   -0.06    │    0.06    │    0    │         │       │
│ 16│ lz_ws2024_xe136_rate_multiplier              │   1.00    │   0.15    │   -0.15    │    0.15    │    0    │         │       │
│ 17│ lz_ws2024_xe127_xe125_rate_multiplier        │   1.01    │   0.19    │   -0.19    │    0.19    │    0    │         │       │
│ 18│ lz_ws2024_ac_rate_multiplier                 │   0.94    │   0.21    │   -0.21    │    0.21    │    0    │         │       │
│ 19│ lz_ws2024_xe124_rate_multiplier              │   1.11    │   0.18    │   -0.18    │    0.18    │    0    │         │       │
│ 20│ lz_ws2024_ll_quenching_factor                │   0.687   │   0.030   

In [ ]:
# lz_ws2024_st.run_inference(output_file_name="lz_ws2024_wimp_ci.csv")
diamx.run_inference_pool(
    lz_ws2024_st,
    output_file_name="lz_ws2024_wimp_ci.csv",
    processes=8,
)
ci = np.loadtxt("./diamx_output/lz_ws2024_wimp_ci.csv", delimiter=",")
plt.plot(ci[:, 0], ci[:, 2] * 1e-44, label="diamx")
plt.yscale("log")
plt.xscale("log")
ci_literature = np.loadtxt("../diamx/data/lz_ws2024_wimp_ci.csv", delimiter=",")
plt.plot(ci_literature[:, 0], ci_literature[:, 1], label="literature")
plt.legend()
plt.yscale("log")
plt.xscale("log")
plt.xlabel("WIMP mass [GeV]")
plt.ylabel("Cross section [cm^2]")
plt.title("LZ WS2024 WIMP 90% CI")
plt.savefig("../plots/lz_ws2024_wimp_ci.png", dpi=600)

## 5. PandaX-4T combined Run0 & Run1

Combined Run0+Run1 spin-independent WIMP limit. The signal-response model follows PRD 110 023029 and the backgrounds/exposures follow PRL 134 011805 Table I (config `pandax4t_run01_wimp_config.json`). Background rates in the config are in **events/year** and are scaled by each run's livetime (exposure = fiducial_mass x livetime = 0.54 / 1.00 tonne-year). The result is compared against the digitized published limit (PRL Fig. 3, `../diamx/data/pandax4t_run01_wimp_ci.csv`).

Note: the reproduced limit matches the published *shape* (minimum near 40 GeV/$c^2$) but its normalization is currently ~3.5x higher. This traces to the NR band sitting ~0.1 too high in $\log_{10}(cS2_b/cS1)$ -- i.e. the NR charge/light yield -- whereas the detector/reconstruction chain and selection efficiency are validated against the data ER band.

In [ ]:
pandax4t_st = diamx.Context(
    os.path.join(diamx_dir, "..", "config", "pandax4t_run01_wimp_config.json"),
)
pandax4t_st.register_experiment(diamx.experiments.PandaX4TRun0)
pandax4t_st.register_experiment(diamx.experiments.PandaX4TRun1)
pandax4t_st.generate_templates()

In [ ]:
# Reproducing Fig. 3 (upper limit)
# pandax4t_st.run_inference(output_file_name="pandax4t_run01_wimp_ci.csv")
diamx.run_inference_pool(
    pandax4t_st,
    output_file_name="pandax4t_run01_wimp_ci.csv",
    processes=8,
)
ci = np.loadtxt("./diamx_output/pandax4t_run01_wimp_ci.csv", delimiter=",")
plt.plot(ci[:, 0], ci[:, 2] * 1e-44, label="diamx")
ci_literature = np.loadtxt("../diamx/data/pandax4t_run01_wimp_ci.csv", delimiter=",")
plt.plot(ci_literature[:, 0], ci_literature[:, 1], label="literature")
plt.legend()
plt.yscale("log")
plt.xscale("log")
plt.xlabel("WIMP mass [GeV]")
plt.ylabel("Cross section [cm^2]")
plt.title("PandaX-4T Run0&1 WIMP 90% CI")
plt.savefig("../plots/pandax4t_run01_wimp_ci.png", dpi=600)